# Question 2: 311 Service Requests Analysis

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Data Source:** NYC OpenData 311 Service Requests (API)
**Dataset:** https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9

**Filters applied:**
1. `created_date` between 12/15/2021 and 3/15/2022 (inclusive)
2. `agency_name` = NYPD or HPD
3. `complaint_type` = noise (all subcategories) or illegal parking

**Key finding from Phase 1:** HPD returned zero matching rows because HPD handles housing quality (heat/hot water, plumbing), not noise or parking. All 198,158 rows are NYPD.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv('../data/311_complaints.csv')
df['created_date'] = pd.to_datetime(df['created_date'])
print(f'Total rows: {len(df):,}')
print(f'Date range: {df["created_date"].min()} to {df["created_date"].max()}')
print(f'Agencies: {df["agency_name"].unique()}')
print(f'Complaint types: {sorted(df["complaint_type"].unique())}')

Total rows: 198,158
Date range: 2021-12-15 00:02:21 to 2022-03-15 23:59:38
Agencies: <StringArray>
['New York City Police Department']
Length: 1, dtype: str
Complaint types: ['Illegal Parking', 'Noise - Commercial', 'Noise - House of Worship', 'Noise - Park', 'Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Vehicle']


---
## Step 3.1–3.6: Query URL (Q2a)

The exact SoQL query URL used to pull this data:

In [2]:
QUERY_URL = (
    'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
    '?$select=unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'
    '&$where=created_date between \'2021-12-15T00:00:00\' and \'2022-03-15T23:59:59\' '
    'AND (agency_name=\'New York City Police Department\' '
    'OR agency_name=\'Department of Housing Preservation and Development\') '
    'AND (starts_with(complaint_type, \'Noise\') '
    'OR complaint_type=\'Illegal Parking\')'
    '&$limit=500000'
)
print(QUERY_URL)
print(f'\nData pulled: {len(df):,} rows | All NYPD (HPD = 0 matching rows)')

https://data.cityofnewyork.us/resource/erm2-nwe9.json?$select=unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude&$where=created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' AND (agency_name='New York City Police Department' OR agency_name='Department of Housing Preservation and Development') AND (starts_with(complaint_type, 'Noise') OR complaint_type='Illegal Parking')&$limit=500000

Data pulled: 198,158 rows | All NYPD (HPD = 0 matching rows)


### Why `starts_with(complaint_type, 'Noise')`?

The 311 system categorizes noise into subcategories: "Noise - Residential", "Noise - Commercial", "Noise - Vehicle", "Noise - Street/Sidewalk", etc. An exact match for `complaint_type = 'Noise'` would only capture the 10,789 records handled by the Department of Environmental Protection — **missing 92% of noise complaints**, which are in NYPD-handled subcategories. Using `starts_with` captures the full scope of noise-related complaints as the question intends.

Additionally, the exact "Noise" type belongs to the **Department of Environmental Protection** (DEP), not NYPD or HPD, so it is correctly excluded by the agency filter.

---
## Step 3.9–3.11: Complaint Counts by Agency and Type (Q2b)

In [3]:
crosstab = pd.crosstab(df['agency_name'], df['complaint_type'], margins=True)

noise_mask = df['complaint_type'].str.startswith('Noise')
summary = pd.DataFrame({
    'Count': df['complaint_type'].value_counts(),
    '% of Total': (df['complaint_type'].value_counts() / len(df) * 100).round(2)
})
summary.loc['TOTAL'] = [len(df), 100.00]
display(summary.style.format({'Count': '{:,.0f}', '% of Total': '{:.2f}%'}))

print(f'\nNoise (all subcategories): {noise_mask.sum():,} ({noise_mask.sum()/len(df)*100:.1f}%)')
print(f'Illegal Parking: {(~noise_mask).sum():,} ({(~noise_mask).sum()/len(df)*100:.1f}%)')
print(f'\nAll {len(df):,} complaints are from NYPD. HPD = 0 matching rows (see Phase 1).')

,Count,% of Total
complaint_type,,
Illegal Parking,"86,493",43.65%
Noise - Residential,"75,543",38.12%
Noise - Street/Sidewalk,"12,743",6.43%
Noise - Vehicle,"11,372",5.74%
Noise - Commercial,"11,114",5.61%
Noise - Park,498,0.25%
Noise - House of Worship,395,0.20%
TOTAL,"198,158",100.00%



Noise (all subcategories): 111,665 (56.4%)
Illegal Parking: 86,493 (43.6%)

All 198,158 complaints are from NYPD. HPD = 0 matching rows (see Phase 1).


counts = df['complaint_type'].value_counts().sort_values(ascending=True)
short_labels = {'Illegal Parking': 'Illegal Parking',
    'Noise - Residential': 'Noise: Residential',
    'Noise - Street/Sidewalk': 'Noise: Street/Sidewalk',
    'Noise - Vehicle': 'Noise: Vehicle',
    'Noise - Commercial': 'Noise: Commercial',
    'Noise - Park': 'Noise: Park',
    'Noise - House of Worship': 'Noise: House of Worship'}

colors = ['#1565c0' if 'Illegal' in ct else '#d32f2f' if 'Residential' in ct else '#ef5350' for ct in counts.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=[short_labels.get(ct, ct) for ct in counts.index],
    x=counts.values, orientation='h', marker_color=colors,
    text=[f'{v:,}' for v in counts.values],
    textposition='outside', textfont_size=11,
))

fig.update_layout(
    title='311 Complaints by Type: NYPD (Dec 15, 2021 – Mar 15, 2022)<br><sup>Total: 198,158 complaints | All NYPD</sup>',
    xaxis_title='Number of Complaints', yaxis_title='',
    height=400, margin=dict(l=160), font=dict(size=10),
    annotations=[dict(x=0.85, y=0.05, xref='paper', yref='paper',
        text='Red = Noise | Blue = Illegal Parking',
        showarrow=False, font=dict(size=10, color='#666'))]
)
fig.show()

In [4]:
counts = df['complaint_type'].value_counts().sort_values(ascending=True)

colors = []
for ct in counts.index:
    if ct == 'Illegal Parking':
        colors.append('#1565c0')
    elif 'Residential' in ct:
        colors.append('#d32f2f')
    else:
        colors.append('#ef5350')

fig = go.Figure()
fig.add_trace(go.Bar(
    y=counts.index,
    x=counts.values,
    orientation='h',
    marker_color=colors,
    text=[f'{v:,}' for v in counts.values],
    textposition='outside',
    textfont_size=11,
))

fig.update_layout(
    title='311 Complaints by Type: NYPD (Dec 15, 2021 – Mar 15, 2022)<br>'
          '<sup>Total: 198,158 complaints | All NYPD (HPD = 0 matching complaints)</sup>',
    xaxis_title='Number of Complaints',
    yaxis_title='',
    height=500,
    width=950,
    margin=dict(l=200),
    font=dict(size=11),
    annotations=[
        dict(x=0.85, y=0.05, xref='paper', yref='paper',
             text='Red = Noise subcategories | Blue = Illegal Parking',
             showarrow=False, font=dict(size=10, color='#666'))
    ]
)
fig.show()

df['date_bin'] = df['created_date'].dt.to_period('W').astype(str)
weekly = df.groupby(['date_bin', 'complaint_type']).size().unstack(fill_value=0)
noise_cols = [c for c in weekly.columns if c.startswith('Noise')]
weekly['Total Noise'] = weekly[noise_cols].sum(axis=1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Total Noise'],
    name='Noise (all subcategories)', mode='lines',
    line=dict(color='#d32f2f', width=2)
))
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Illegal Parking'],
    name='Illegal Parking', mode='lines',
    line=dict(color='#1565c0', width=2)
))

fig2.update_layout(
    title='Weekly Complaint Trends: Noise vs Illegal Parking<br><sup>NYPD, Dec 2021 – Mar 2022</sup>',
    xaxis_title='Week', yaxis_title='Complaints per Week',
    height=350, font=dict(size=10), legend=dict(x=0.01, y=0.99),
)
fig2.show()

In [5]:
df['week'] = df['created_date'].dt.isocalendar().week.astype(int)
df['year_week'] = df['created_date'].dt.strftime('%Y-W%U')
df['date_bin'] = df['created_date'].dt.to_period('W').astype(str)

weekly = df.groupby(['date_bin', 'complaint_type']).size().unstack(fill_value=0)

noise_cols = [c for c in weekly.columns if c.startswith('Noise')]
weekly['Total Noise'] = weekly[noise_cols].sum(axis=1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Total Noise'],
    name='Noise (all subcategories)', mode='lines',
    line=dict(color='#d32f2f', width=2)
))
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Illegal Parking'],
    name='Illegal Parking', mode='lines',
    line=dict(color='#1565c0', width=2)
))

fig2.update_layout(
    title='Weekly Complaint Trends: Noise vs Illegal Parking<br>'
          '<sup>NYPD, Dec 2021 – Mar 2022 | Note seasonal patterns</sup>',
    xaxis_title='Week',
    yaxis_title='Complaints per Week',
    height=400,
    width=950,
    font=dict(size=11),
    legend=dict(x=0.01, y=0.99),
)
fig2.show()

### Policy Message

**What policymakers should know:**

Over the 3-month period from December 2021 to March 2022, the NYPD received nearly **200,000 noise and illegal parking complaints** — an extraordinary volume that represents a significant allocation of police resources to quality-of-life issues.

Three findings stand out:

1. **Noise - Residential dominates.** With 75,543 complaints (38% of all complaints), residential noise is by far the single largest quality-of-life concern reported to 311. This suggests that dense residential living conditions, particularly in multi-unit buildings, are a persistent source of friction between neighbors.

2. **Illegal parking is nearly as large as all noise combined.** At 86,493 complaints (44%), illegal parking rivals the entire category of noise complaints (111,665). This points to a significant parking enforcement burden on NYPD and raises the question of whether parking enforcement could be handled by a civilian agency, freeing police resources for higher-priority public safety matters.

3. **HPD had zero matching complaints.** The question asked us to pull data for both NYPD and HPD, but HPD does not handle noise or parking complaints — they handle housing quality issues like heat/hot water and plumbing. This is worth noting because it highlights the specialized division of responsibilities among NYC agencies, and suggests that inter-agency coordination may be needed when complaints span jurisdictional boundaries.

For policymakers, the key takeaway is that **quality-of-life enforcement consumes substantial NYPD resources**. If the city wants to rethink policing priorities, understanding the volume and nature of 311 complaints is an essential starting point.

---
## Step 3.15–3.18: Research Question (Q2d)

### Research Question

**"Do neighborhoods with lower median household incomes generate more noise and illegal parking complaints per capita, after controlling for population density?"**

### Additional Data to Merge

To answer this question, we would merge the 311 complaint data with:

1. **American Community Survey (ACS) 5-Year Estimates** (U.S. Census Bureau)
   - Median household income by zip code or community district
   - Total population by zip code or community district
   - Land area (to compute population density)
   - Available via: https://data.census.gov or the Census API

2. **Merge key:** The 311 data contains `incident_zip` and `borough` fields. ACS data is available at the zip code tabulation area (ZCTA) level, which closely aligns with postal zip codes.

### Equation

The relationship can be expressed as a simple linear regression:

> **ComplaintsPerCapita_i = β₀ + β₁ · MedianIncome_i + β₂ · PopulationDensity_i + ε_i**

Where:
- **ComplaintsPerCapita_i** = total noise + parking complaints / population in area i
- **MedianIncome_i** = median household income in area i
- **PopulationDensity_i** = population / land area in area i
- **β₀** = intercept
- **β₁** = the association between income and complaints per capita (key coefficient of interest)
- **β₂** = the association between density and complaints per capita (control variable)
- **ε_i** = error term

If β₁ is negative and statistically significant, it would indicate that lower-income neighborhoods file more complaints per person, after accounting for how densely populated they are.

### Why Is This Important for Policymaking?

This relationship matters for two reasons:

**1. Resource allocation equity.** If lower-income neighborhoods generate more complaints per capita, it could indicate that these areas face greater quality-of-life challenges (noise from overcrowded housing, illegal parking from inadequate infrastructure). City services should be proportionally directed to where need is greatest, not just where complaints are loudest.

**2. Complaint-driven vs. need-driven governance.** 311 data reflects who files complaints, not necessarily who experiences the most problems. Higher-income residents may be more likely to file complaints, biasing resource allocation toward wealthier neighborhoods. Understanding this relationship helps policymakers distinguish between actual need and reporting behavior, ensuring that city services address underlying conditions rather than just responding to the most vocal complainants.

Studying this relationship could inform whether NYC's 311-driven service model equitably serves all communities, or whether supplementary outreach is needed in under-reporting neighborhoods.

---
## Audit Gate 3 Checklist

| Check | Status |
|-------|--------|
| Exact query URL is a valid https://... link | ✅ PASS (parenthesization corrected) |
| Date boundaries inclusive on both ends | ✅ PASS: min=12/15/2021, max=3/15/2022 |
| $limit set and not truncated | ✅ PASS: 198,158 < 500,000 |
| Cross-tab spot-checked | ✅ PASS: Illegal Parking = 86,493 matches |
| Both description AND visualization present for Q2c | ✅ PASS |
| Equation is genuinely simple | ✅ PASS: single OLS regression |
| Policy message grounded in observed data | ✅ PASS |
| Noise matching approach documented | ✅ PASS |
| HPD zero finding explained | ✅ PASS |